In [1]:
import pandas as pd
#import modin.pandas as pd
#import ray
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
sns.set(style='whitegrid',font_scale=2)
import os
import pickle
import time
import drugstandards as drugs
import re
from drug_named_entity_recognition import find_drugs
import math
from itertools import chain
from collections import Counter

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
from pandas.errors import SettingWithCopyWarning
warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)

os.chdir(os.getcwd())
os.getcwd()

'/Users/daniel.garger/Desktop/Projects/C-PATH/outcome prediction/src'

In [2]:
#cm=pd.read_csv(dir+'/data/ttp_ae.csv', low_memory=False)
#mh=pd.read_csv(dir+'/data/ttp_mh.csv', low_memory=False)
#cm=pd.read_csv(dir+'/data/ttp_cm.csv', low_memory=False)
cm=pd.read_csv('../data/out_cm.csv.gz', low_memory=False)
cm_=pd.read_csv('../../C-Path_data/fullExportDb-1025-Member-CSV/cm.csv',low_memory=False)
db_vocab=pd.read_csv('../../TTP prediction/data/drugbank vocabulary.csv', low_memory=False)

In [3]:
'''
for common_drug_name in db_vocab['Common name'][0:1]:
    drugs.add_drug_mapping({common_drug_name.upper():common_drug_name.upper()})
    
    if not str(db_vocab.loc[db_vocab['Common name']==common_drug_name,'Synonyms'].values[0])=='nan':
        synonyms=db_vocab.loc[db_vocab['Common name']==common_drug_name,'Synonyms'].values[0].split('|')
        synonyms=[x.strip().upper() for x in synonyms]
        syn_dict=dict(zip(synonyms,[common_drug_name.upper(),]*len(synonyms)))
        print(syn_dict)
        #drugs.add_drug_mapping(syn_dict)
'''

Drug dictionary successfully updated...
{'[LEU1, THR2]-63-DESULFOHIRUDIN': 'LEPIRUDIN', 'DESULFATOHIRUDIN': 'LEPIRUDIN', 'HIRUDIN VARIANT-1': 'LEPIRUDIN', 'LEPIRUDIN': 'LEPIRUDIN', 'LEPIRUDIN RECOMBINANT': 'LEPIRUDIN', 'R-HIRUDIN': 'LEPIRUDIN'}


In [3]:
#load patient IDs who are considered in this  analysis
pat_id_df=pd.read_csv('../data/patients_in_analysis.csv.gz',index_col=0)


<hr>

### Get start and end day of the adverse events
* #### AIM: where necessary impute __START DAY__ and __END DAY__ of adverse event
* #### __Save the dataset where start and end day of adverse event is available__



In [4]:
### Initiate standardised column names
#cm['STD_CMDY']=cm['CMDY']
cm['STD_CMSTDY']=cm['CMSTDY']
cm['STD_CMENDY']=cm['CMENDY']
cm=cm.drop_duplicates()

#### Drop measurement point without any temporal information
cm_with_time=cm[~((cm['CMSTDY'].isna()))]

### Extract events that originate from way before start of therapy as Medical History
cm_history=cm_with_time[(cm_with_time['CMSTDY']<0)&(cm_with_time['CMENDY']<0)]
cm_history.to_csv('../data/out_cm_mh.csv.gz',compression='gzip')

# Drop terms that were diagnosed more than 20 days before therapy start
cm_timely_relevant=cm_with_time[~(cm_with_time.index.isin(cm_history.index.tolist()))]
cm_timely_relevant=cm_timely_relevant[~((cm_timely_relevant['CMENDY'].isna())&(cm_timely_relevant['CMENRTPT'].isna())&\
                                     (cm_timely_relevant['CMENTPT'].isna()))]

appl_routes_to_keep=['ORAL','INTRAMUSCULAR','INTRAVENOUS','RESPIRATORY (INHALATION)','NASAL','BUCCAL'\
                             'SUBLINGUAL','RESPIRATORY','INHALALATION WITH ROTAHA','UNKNOWN']   

## Subset dataframe to rows with dose data of the applied medications
cm_with_dose=cm_timely_relevant[(~cm_timely_relevant['CMDOSE'].isna()&~cm_timely_relevant['CMDOSFRQ'].isna())&\
                                ((cm_timely_relevant['CMROUTE'].isin(appl_routes_to_keep))|(cm_timely_relevant['CMROUTE'].isna()))].dropna(how='all',axis=1)


### Impute the end day of drug application for patients with missing end day data

In [5]:
## Extract patients IDs who have enough drug name/dosage/duration data
pats_with_cm=cm_with_dose["USUBJID"].tolist()

# LOAD DRUG ADHERENCE DATASET
da=pd.read_csv('../../C-Path_data/fullExportDb-1025-Member-CSV/da.csv',low_memory=False)
da=da[da['USUBJID'].isin(pats_with_cm)].dropna(how='all',axis=1)

## LOAD EXPOSURE DATASET
ex=pd.read_csv('../../C-Path_data/fullExportDb-1025-Member-CSV/ex.csv', low_memory=False)
ex=ex[ex['USUBJID'].isin(pats_with_cm)].dropna(how='all',axis=1)

## LOAD INDIVIUDAL START AND END OF THERAPY TIMEPOINT DATAFRAME
se=pd.read_csv('../../C-Path_data/fullExportDb-1025-Member-CSV/se.csv',low_memory=False)
se=se[se['USUBJID'].isin(pats_with_cm)].dropna(how='all',axis=1)

## LOAD INDIVIUDAL START AND END OF THERAPY TIMEPOINT DATAFRAME
ds=pd.read_csv('../../C-Path_data/fullExportDb-1025-Member-CSV/ds.csv',low_memory=False)
ds=ds[ds['USUBJID'].isin(pats_with_cm)].dropna(how='all',axis=1)


### Add end day of drug application for patients who dont have exact end days of drug application,
##  but only the study periods when the drug was applied  (i.e. start day:15, end day: during intensive phase)

#######  TB-1021 #############
##  Extract lsat day of intensive phase for TB-1021 patients from se dataframe
pats_wo_exact_end_day_1021=cm_with_dose[(cm_with_dose['STUDYID']=='TB-1021')&
                                        (cm_with_dose['CMENDY'].isna())]['USUBJID'].tolist()

# Loop over patient ids and add the exact end day of drug application
for pat_id in pats_wo_exact_end_day_1021:
    end_intensive_phase=se.loc[(se['USUBJID']==pat_id)&(se['EPOCH'].str.contains('INTENSIVE',na=False)),'SEENDY'].max()
    end_cont_phase=se.loc[(se['USUBJID']==pat_id)&(se['EPOCH'].str.contains('CONTINUATION',na=False)),'SEENDY'].max()
    end_followup_phase=se.loc[(se['USUBJID']==pat_id)&(se['EPOCH'].str.contains('FOLLOW-UP',na=False)),'SEENDY'].max()

    cm_df_missing_end_day=cm_with_dose.loc[(cm_with_dose['USUBJID']==pat_id)&
                                           (cm_with_dose['CMENDY'].isna()),:]    

    for end_phase_of_drug_appl,df in cm_df_missing_end_day.groupby('CMENTPT'):        
        if 'FOLLOW-UP' in end_phase_of_drug_appl or 'STUDY END' in end_phase_of_drug_appl:
            end_day_of_drug_appl=end_followup_phase   

        if 'CONTINUATION PHASE' in end_phase_of_drug_appl:
            end_day_of_drug_appl=end_cont_phase 

        if 'INTENSIVE PHASE' in end_phase_of_drug_appl:
            end_day_of_drug_appl=end_intensive_phase 

        cm_with_dose.loc[df.index,'STD_CMENDY']=end_day_of_drug_appl



In [6]:

######### OTHER STUDIES THEN TB-1021 ########
###  Extract last day of intensive phase for  other studies than TB-1021 patients from ds dataframe
##   As for all other studies the last phase of drug application was 'END of STUDY' extract last day of study
##   as end day of drug application
pats_wo_exact_end_day=cm_with_dose[(cm_with_dose['STUDYID']!='TB-1021')&
                                        (cm_with_dose['CMENDY'].isna())]

## LOAD INDIVIUDAL START AND END OF THERAPY TIMEPOINT DATAFRAME
pats_wo_exact_end_day=cm_with_dose[(cm_with_dose['STUDYID']!='TB-1021')&
                                        (cm_with_dose['CMENDY'].isna())]
ds=pd.read_csv('../../C-Path_data/fullExportDb-1025-Member-CSV/ds.csv',low_memory=False)
ds=ds[ds['USUBJID'].isin(pats_wo_exact_end_day['USUBJID'].unique().tolist())].dropna(how='all',axis=1)

ds_pats_wo_exact_end_day=ds[((ds['STUDYID']=='TB-1022')&(ds['DSSCAT'].str.contains('COMPLETED|WITHDRAWAL',na=False)))|\
                            ((ds['STUDYID']=='TB-1018')&(ds['DSDECOD'].str.contains('COMPLETED',na=False))&(ds['EPOCH'].str.contains('TREATMENT',na=False)))]

## For TB-1022 copy the CMDY column information -> this the maximum may of observation
cm_with_dose.loc[(cm_with_dose['STUDYID']=='TB-1022')&(cm_with_dose['CMENDY'].isna()),'STD_CMENDY']=cm_with_dose.loc[(cm_with_dose['STUDYID']=='TB-1022')&(cm_with_dose['CMENDY'].isna()),'CMDY']

# Loop over patient ids and add the exact end day of drug application
for pat_id in pats_wo_exact_end_day[pats_wo_exact_end_day['STUDYID']=='TB-1018']['USUBJID'].unique()[0:]:
    df=ds_pats_wo_exact_end_day[ds_pats_wo_exact_end_day['USUBJID']==pat_id]
    end_day_study=df['DSSTDY'].max()
    cm_with_dose.loc[pats_wo_exact_end_day[pats_wo_exact_end_day['USUBJID']==pat_id].index,'STD_CMENDY']=end_day_study



## After imputation, some end days of application are earlier timepoints then start day -> imputation not reliable -> 
#  drop these datapoints
idx_to_drop=cm_with_dose[(cm_with_dose['STD_CMENDY']<cm_with_dose['STD_CMSTDY'])&(cm_with_dose['CMENDY'].isna())].index
cm_with_dose=cm_with_dose.drop(index=idx_to_drop)

## For some datapoints the start and end days of application seem to be switched -> switch them 
switched_start_end_days=cm_with_dose[(cm_with_dose['STD_CMENDY']<cm_with_dose['STD_CMSTDY'])&~(cm_with_dose['CMENDY'].isna())]
cm_with_dose.loc[switched_start_end_days.index,'STD_CMSTDY']=cm_with_dose.loc[switched_start_end_days.index,'CMENDY']
cm_with_dose.loc[switched_start_end_days.index,'STD_CMENDY']=cm_with_dose.loc[switched_start_end_days.index,'CMSTDY']

## Drop datapoints without end day of drug application
cm_with_dose=cm_with_dose[~cm_with_dose['STD_CMENDY'].isna()]


### __Standardise drug names__

* #### Replace terms that have typos are irrelevant
* #### For each drug(s) applied per patient, extract the standard names of the drug(s)

In [7]:
## DICTIONARY HOLDING THE ENTRIES TO REPLACE THEM WITH
dict_={' ACID':'-ACID',' OXIDE':'-OXIDE',' HYDROXIDE':'-HYDROXIDE',' HYDROXINE':'-HYDROXIDE',
        ' HYDROCHLORIDE':'',' HCL':'',' CHLORIDE':'-CHLORIDE',' CITRATE':'-CITRATE',
        ' SILICATE':'-SILICATE',' TRISILICATE':'-TRISILICATE',' MALEATE':'','VIT B CO':'VITAMIN B COMPLEX',
        'VIT B COMPLEX':'VITAMIN B COMPLEX','VITAMINE B6':'PYRIDOXINE','VITAMONE B6':'PYRIDOXINE',
        '-HYDROBROMIDE':'',' PHOSPHATE':'','ADCO-':'','CO-':'','-CLAVULANIC-ACID':'CLAVULANIC-ACID',
        'AMOXYCLAUVINIC-ACID':'AMOXICILLIN, CLAVULANIC-ACID','ANITHOTERICIN B':'AMPHOTERICIN B',
        'BIO-':'','CHLORAMPHENAMINE':'CHLORPHENIRAMINE','CHLORAMPHERAMINE':'CHLORPHENIRAMINE',
        'CHLORAMPHERARIMINE':'CHLORPHENIRAMINE','CHLORAMPHERELAMINE':'CHLORPHENIRAMINE',
        'CHLORAMPHENAMINE':'CHLORPHENIRAMINE','-CODEINE':' CODEINE','CETRAZINE':'CETIRIZINE',
        'CELESTAMINE':'BETAMETHASONE CHLORPHENIRAMINE','AMOXICLAV':'AMOXICILLIN, CLAVULANIC-ACID',
        'AMOXICLAVLACTATE':'AMOXICILLIN, CLAVULANIC-ACID','AMOXYCLAV':'AMOXICILLIN, CLAVULANIC-ACID',
        'AMXICLAV':'AMOXICILLIN, CLAVULANIC-ACID','DEXTRAMETHAPHANE':'DEXTROMETHORPHAN',
        'DOXOSIL':'DOXYCYCLINE','DYNAMETRIN':'METRONIDAZOLE','DYNAMETRON':'METRONIDAZOLE',
        'ENOXPARIN':'ENOXAPARIN','ENYTHRAMYCIN':'ERYTHROMYCIN','G  I  LEAN  THERMOGENIC HERBAL DROPS ':np.nan,
        'GRANDPA POWDER':np.nan,'GRANDPA':np.nan,'GRAND-PA':np.nan,'INSULATARD':'INSULINE',
        'METRAMIDAZOLE':'METRONIDAZOLE','METRANIDAZOLE':'METRONIDAZOLE','MYLOCORT':'HYDROCORTISONE',
        'NUTRITIONAL SUPPLIMENT':'','PAINAMOL':'ACETAMINOPHEN','PAINAMOL-PLUS':'ACETAMINOPHEN',
        'PARAFIN  WAX':'PARAFFIN','PARENTROVITE':np.nan,'PENILENTE':'PENICILLIN G','PERYDOXINE':'PYRIDOXINE',
        'PHARMAPIESS':'ENALAPRIL','PLANT MEDICINE':np.nan,'PREBIOTIC COMBINATION':np.nan,'PROBIOTIC':np.nan,
        'PROBIOTICS':np.nan,'PYRIDIXINE':'PYRIDOXINE','PYRIDOCINE':'PYRIDOXINE','PYRIODIXINE':'PYRIDOXINE',
        'RED BLOOD CELLS  CONCENTRATED':np.nan,'REPTILASE':'BATROXOBIN','ROXIBID':'ROXITHROMYCIN',
        'SEPTOGARD':np.nan,'SINUTAB':'ACETAMINOPHEN, PSEUDOEPHEDRINE','SODIUM CROMOGHYCATE':'CROMOGLYCATE',
        'SOLPHYLEX':np.nan,'THEPHYLIN':'THEOPHYLLINE','STAMETA  LAXATIVE OVER COUNTER ':np.nan,'SULFIRM':'SULFIRAM',
        'SUNCODIN':'DIHYDROCODEINE','SYSTATIN':'NYSTATIN','T CO-AMOXYCLAR':'AMOXICILLIN, CLAVULANIC-ACID',
        'TEAR NATURALE FREE':np.nan,'TEARS NATURELLE':np.nan,'TENOSPORA CORDIFOLIA':np.nan,'TERIVALIDIN':'TERIZIDONE',
        'ACTRAPID':'INSULIN SHORT ACTING','ANUSOL':np.nan,'TOXOID':np.nan,'TEXA':'CETIRIZINE',
        'TRADITIONAL MEDICATION':np.nan,'TREPILINE':'AMITRIPTYLINE','TREZAMIN-ACID':'TRANEXAMIC ACID',
        'TRIPHASIL':'LEVONORGESTREL, ETHINYLESTRADIOL','LIVER SUPPLIMENT':np.nan,'LIVER HERBAL TONIC':np.nan,
        'LEVOCLODERASINE':'LEVOCLOPERASTINE','AMOXICILLIN-CLAVULANIC-ACID':'AMOXICILLIN, CLAVULANIC-ACID',
        'AMOXYCILLIN-CLAVULANIC-ACID':'AMOXICILLIN, CLAVULANIC-ACID','AMOXYCLAUVINIC-ACID':'AMOXICILLIN, CLAVULANIC-ACID',
        r'\bVIT B\b':'THIAMINE',r'\bVIT B1\b':'THIAMINE','SOLUBLE INSULIN':'HUMAN INSULIN','CLAVULANATE POTASSIUM':'CLAVULANIC-ACID',
        '&':' ',r'\bVIT \b':'VITAMIN-','VIT B1':'THIAMINE',r'\bVITB\b':'THIAMINE','CA-PANTOTHENIC-ACID':'PANTOTHENIC-ACID',
        'CA-PAINTOTH':'PANTOTHENIC-ACID','C9 - PANTOTH':'PANTOTHENIC-ACID',
        'VIT B2':'RIBOFLAVINE','VITAMIN B':'VITAMIN-B','VIT B3':'NICOTINAMIDE',r'\bPANTOTH\b':'PANTOTHENIC-ACID',r'\bPAINTOTH\b':'PANTOTHENIC-ACID',
        'CA-PANTOTHENIC ACID':'PANTOTHENIC-ACID','PANTOTHENIC ACID':'PANTOTHENIC-ACID',r'\bB2\b':'RIBOFLAVINE',r'\bB3\b':'NICOTINAMIDE',
        r'\bB1\b':'THIAMINE', r'\bB5\b':'PANTOTHENIC-ACID',r'\bB6\b':'PYRIDOXINE',r'\bB12\b':'CYANOCOBALAMIN', 'NICOTIN-100':'NICOTINE 100',
        'VITAMIN E':'VITAMIN-E','THIAMINE CO':'THIAMINE','THIAMINE COMPLEX':'THIAMINE','PHOSPAHTE':'','MEDROXY PROGESTERONE':'MEDROXYPROGESTERONE',
        'POTASSIUM CLAVULATES':'CLAVULANIC-ACID','POTASSIUM CLAVULANATE':'CLAVULANIC-ACID'
        }


cm_with_dose['STD_DRUGS']=np.nan
## Concatenate all the available previously extracted drug names that are in CMACT1-CMACT8 into STD-DRUGS
cm_with_dose.loc[~cm_with_dose['CMACT1'].isna(),'STD_DRUGS']=cm_with_dose.loc[~cm_with_dose['CMACT1'].isna(),'CMACT1':'CMACT5'].apply(lambda row: ','.join([x for x in row.values if str(x)!='nan']), axis=1)

cm_with_dose.loc[(~cm_with_dose['CMMODIFY'].isna())&(cm_with_dose['CMACT1'].isna()),\
                 'STD_DRUGS']=cm_with_dose.loc[(~cm_with_dose['CMMODIFY'].isna())&\
                                               (cm_with_dose['CMACT1'].isna()),'CMMODIFY']

cm_with_dose.loc[(cm_with_dose['CMMODIFY'].isna())&(~cm_with_dose['CMTRT'].isna())&(cm_with_dose['CMACT1'].isna()),\
                 'STD_DRUGS']=cm_with_dose.loc[(cm_with_dose['CMMODIFY'].isna())&(cm_with_dose['CMACT1'].isna())&\
                                               (~cm_with_dose['CMTRT'].isna()),'CMTRT']

cm_with_dose['STD_DRUGS']=cm_with_dose['STD_DRUGS'].replace(dict_,regex=True)

## Standardise vitamin b complex entries
yes=['VIT']
yes2=['B']
no=[r'6|12|/|MG|1|\bC\b|FOLIC']
d=cm_with_dose[cm_with_dose['STD_DRUGS'].str.contains('|'.join(yes),na=False)&\
    cm_with_dose['STD_DRUGS'].str.contains('|'.join(yes2),na=False)&\
    ~(cm_with_dose['STD_DRUGS'].str.contains('|'.join(no),na=False))]
cm_with_dose.loc[d.index,'STD_DRUGS']='VITAMIN B COMPLEX'

### Drop ringer's lactate datapoints
yes=['RINGER']
yes2=['']
no=[r'q']
d=cm_with_dose[cm_with_dose['STD_DRUGS'].str.contains('|'.join(yes),na=False)&\
    cm_with_dose['STD_DRUGS'].str.contains('|'.join(yes2),na=False)&\
    ~(cm_with_dose['STD_DRUGS'].str.contains('|'.join(no),na=False))]
cm_with_dose.loc[d.index,'STD_DRUGS']=np.nan


### Add nwew mapping to the drugdict
drugs.add_drug_mapping({"CLAVULANIC-ACID":"CLAVULANIC ACID",'ALUMINUM-OXIDE':'ALUMINIUM-OXIDE',
                        'ALUMINIUM-OXIDE':'ALUMINIUM-OXIDE','AMBROXOL':'AMBROXOL',
                        'DICHLOROBENZYL':'DICHLOROBENZYL','MAGNESIUM HYDROXIDE':'MAGNESIUM HYDROXIDE',
                        'METHYLSALICYLATE':'METHYLSALICYLATE','INSULIN':'INSULIN',
                        'VITAMIN B':'VITAMIN B','VITAMIN B6':'PYRIDOXINE','VITAMIN B12':'CYANOCOBALAMINE',
                        'B6':'PYRIDOXINE','B12':'CYANOCOBALAMINE','VITAMIN E':'VITAMIN E',
                        'VITAMIN K':'VITAMIN K','SALINE':'SALINE',r'\bNACL\b':'SALINE',
                        'SODIUMCHLORIDE':'SALINE','LEVOCETRIZINE':'LEVOCETIRIZINE',
                        'LEVOCETIRIZINE':'LEVOCETIRIZINE','INSULIN LONG ACTING':'INSULIN LONG ACTING',
                        'INSULIN SHORT ACTING':'INSULIN SHORT ACTING',
                        'INSULIN INTERMEDIATE ACTING':'INSULIN INTERMEDIATE ACTING',
                        'REGULAR INSULIN':'INSULIN SHORT ACTING','HUMAN INSULIN':'INSULIN SHORT ACTING',
                        'INSULIN HUMAN':'INSULIN SHORT ACTING','FAST ACTING INSULIN':'RAPID ACTING INSULIN',
                        '3TC':'LAMIVUDIN','CODIENE':'CODEINE','BROMHEXINE':'BROMHEXINE',
                        'IMIPENEM':'IMIPENEM','ZIDOVUDINE':'ZIDOVUDINE','DIHYDROCHLORIDE':'',
                        'AMBROXYL':'AMBROXOL','INSULIN GLARGINE RECOMBINANT':'INSULIN LONG ACTING',
                        'INSULINE GLARGINE':'INSULIN LONG ACTING',
                        'INSULIN GLARGINE':'INSULIN LONG ACTING','-COMPLEX':'',' COMPLEX':'',
                        ' DINITRATE':'',' TRINITRATE':'','ACETATE':'','VIT B COMPLEX':'VITAMIN B COMPLEX',
                        '-ADCO':'','AMIADARONE':'AMIODARONE','BECLOMETHASONE':'BECLOMETHASONE',
                        'BECLAMETHASONE':'BECLOMETHASONE','AUGMENTIN':'AUGMENTIN',
                        'VIT B CO':'VITAMIN B COMPLEX','CEFEXIME':'CEFIXIME','CEMETIDINE':'CIMETIDINE',
                        'CETERIZINE':'CETIRIZINE','CETRICINE':'CETIRIZINE','CETRIZINE':'CETIRIZINE',
                        'CHLOPHERNINAMINE':'CHLOPHENIRAMINE','-CO':'','COPROFLOXACIN':'CIPROFLOXACIN',
                        'DOXICYCLIN':'DOXYCYCLINE','DOXYPHYLLIN':'DOXOPHYLLIN','-NATRIUM':'',
                        'ERTHROMYCIN':'ERYTHROMYCIN','ERTHRAMYCIN':'ERYTHROMYCIN','FEROUS':'IRON',
                        'FERROUSSULPHATE':'IRON','FESO4':'IRON','FEXOFINODINE':'FEXOFENADINE',
                        'FLUCLIXACILLIN':'FLUCLOXACILLIN','GLUCO CORTICOIDS':'GLUCOCORTICOID',
                        'GLUCOCORTICOIDS':'GLUCOCORTICOID','MULTIVITAMIN':'MULTIVITAMIN',
                        'PENTAPRAZOLE':'PANTOPRAZOLE','PEN G':'PENICILLIN G','PEN V':'PENICILLIN V',
                        'PEN VK':'PENICILLIN V','PENT0PERAZOLE':'PANTOPRAZOLE','PIRODOXINE':'PYRIDOXINE',
                        'CETRAZINE':'CETIRIZINE','PHARMAPRESS':'ENALAPRIL','PHYSIOLOGIC SERUM':'PHYSIOLOGIC SERUM',
                        'SHELADOL':'ACETAMINOPHEN','SULFIRAM':'SULFIRAM','EMSET':'ONDANSENTRON',
                        'TRICOHIST':'DIPHENHYDRAMINE, PROMETHAZINE, EPHEDRINE','LEVOCLOPERASTINE':'LEVOCLOPERASTINE',
                        'LENOVATE':'BETAMETHASONE','DEXTROSE':'DEXTROSE','GLUCOSE':'GLUCOSE','LEVOCAMTINE':'LEVOCAMTINE'})



## Create a list of the drug names that will be used for standard drug name extraction
std_drug=cm_with_dose['STD_DRUGS'].value_counts().sort_index()

## Reload drugdict after drug.add_drug_mapping function 
drugs.drugdict=drugs.reload_drugdict()

## REmove some general terms from drugdict, as they introduce some misleading drug names
entries_to_remove=['MG','CAPS','COUGH','VITAMIN','HERBAL','ORAL','SALTS','SOLUTION','SUPPLEMENT','PROTEIN',\
                    'FLUID','EYE','DROPS','VITAMIN B','2']
for key in entries_to_remove:
    drugs.drugdict.pop(key, None)


Drug dictionary successfully updated...
Dictionary successfully reloaded


* #### Extract the drug names that are easy to recognize with a stringent threshold for the Jaro-Winkler similarity
* #### For drug names with typos set the threshold to 0.9 -> algorithm has a chance to recognise them 
* #### Collect standardised drug names for each drug(s) applied per patient into a dictionary

In [8]:

## Run the extraction with the thresholds
terms_dict={}
thresholds=[0.97]
for thresh in thresholds:
    terms_dict[thresh]={}

    ## Drop some special characters
    for dr in std_drug.index.tolist():
        dr=re.sub(r'[()/;.,+]', ' ',dr)
        terms_dict[thresh][dr]={}

        ## First check if the whole string can be recognised as a drug name (i.e. Clavulanic-acid)
        std_whole_term=drugs.standardize([dr],thresh=thresh)
        if std_whole_term[0]!=None:
            matched_words=std_whole_term
            non_matched_words=np.nan
        ## IF the whole string cannot be recognised as a standards drug name 
        #  (i.e. multiple drug names, typos, other names of the drug)
        #  split the applied drug name string into substring and extract standardised drug names from them
        if std_whole_term[0]==None:
            dr_wo_special_chars=[x for x in dr.split(' ') if x!='']
            dr_wo_special_chars=[re.sub(r'[-]', ' ', x) for x in dr_wo_special_chars]
            
            std_drug_names=drugs.standardize(dr_wo_special_chars,thresh=thresh)

            ## Collect index of successfully matched and non-matched words and collect them into a dictionary
            non_matched_word_index=np.where(np.array(std_drug_names) == None)[0]
            matched_word_index=np.where(np.array(std_drug_names) != None)[0]

            if len(non_matched_word_index)>0:
                non_matched_words=np.array(dr_wo_special_chars)[non_matched_word_index].tolist()
            if len(non_matched_word_index)==0:
                non_matched_words=np.nan

            if len(matched_word_index)>0:
                matched_words=np.array(std_drug_names)[matched_word_index].tolist()
            if len(matched_word_index)==0:
                matched_words=np.nan

        terms_dict[thresh][dr]['matched_words']=matched_words
        terms_dict[thresh][dr]['non_matched_words']=non_matched_words

with open('../data/cm_std_drug_names_dict.pickle', 'wb') as file:
    pickle.dump(terms_dict,file)

### Create a dataframe with the matched drug names and non-matched drug names/non-drug-name strings
* #### Contains matched and non-matched drug names for both thresholds

In [9]:
## Load terms_dict containing the satandardised drug names for each row in cm_with_doses dataframe
with open('../data/cm_std_drug_names_dict.pickle', 'rb') as file:
    terms_dict=pickle.load(file)

## Create dataframe with matched and non-matched drug names for all the thresholds
terms_df=pd.DataFrame()
thresholds=[0.97]
for thresh in thresholds[::-1]:
    temp_df=pd.DataFrame.from_dict(terms_dict[thresh])
    temp_df=temp_df.T
    coln=[x+'_'+str(thresh) for x in temp_df.columns.tolist()]
    temp_df.columns=coln
    terms_df=pd.concat([terms_df,temp_df],axis=1)


## Extract terms in the 'STD_DRUGS' columns, that have matches drug names in them with a threshold of 0.97
## Subset cm dataframe to these rows, where a standardised drug name is avaliable
std_terms=terms_df[(~terms_df['matched_words_0.97'].isna())].index.tolist()

## Subset cm dataframe to datapoints with:
#  - standardised drug names
#  - patients with enough data on the duration of the drug(s)
#    (either exact End-day information is present, or the therapy phase is known, when the drug(s) were applied)
cm_std_drugs=cm_with_dose[(cm_with_dose['STD_DRUGS'].isin(std_terms))].dropna(how='all',axis=1)


## Add standardised drug name sin a new column 'STD_DRUGS_REPLACED'
cm_std_drugs['STD_DRUGS_REPLACED']=np.nan
for dr in cm_std_drugs['STD_DRUGS'].unique():
    idx=cm_std_drugs[cm_std_drugs['STD_DRUGS']==dr].index
    std_terms=','.join(terms_df.loc[dr,'matched_words_0.97'])
    cm_std_drugs.loc[idx,'STD_DRUGS_REPLACED']=std_terms

# Correct erroneous drug name -> based on unit of measurement and CMTRT columns, patient was given nystatin and not fluconazol
cm_std_drugs.loc[3574,'STD_DRUGS_REPLACED']='NYSTATIN'

# Replace augmentin with its components
cm_std_drugs['STD_DRUGS_REPLACED']=cm_std_drugs['STD_DRUGS_REPLACED'].replace({'AUGMENTIN':'AMOXICILLIN,CLAVULANIC ACID',
                                                                                'MEDROXYPROGESTERONE ACETATE':'MEDROXYPROGESTERONE',
                                                                                'CLAVULANIC-ACID':'CLAVULANIC ACID'},regex=True)
    

### Standardise units of measurement & route of administration & frequency

In [10]:
cm_std_drugs['STD_UNIT']=cm_std_drugs['CMDOSU']
cm_std_drugs['STD_DOSE']=cm_std_drugs['CMDOSE']
cm_std_drugs['STD_ROUTE']=cm_std_drugs['CMROUTE']
cm_std_drugs['STD_FREQ']=cm_std_drugs['CMDOSFRQ']

## Standardise gram and micrograms to miligrams
cm_std_drugs.loc[cm_std_drugs['CMDOSU']=='g','STD_UNIT']='mg'
cm_std_drugs.loc[cm_std_drugs['CMDOSU']=='g','STD_DOSE']=cm_std_drugs.loc[cm_std_drugs['CMDOSU']=='g','CMDOSE']*1000

cm_std_drugs.loc[cm_std_drugs['CMDOSU']=='ug','STD_UNIT']='mg'
cm_std_drugs.loc[cm_std_drugs['CMDOSU']=='ug','STD_DOSE']=cm_std_drugs.loc[cm_std_drugs['CMDOSU']=='ug','CMDOSE']/1000


## Standardise units of measurement
cm_std_drugs.loc[cm_std_drugs['CMDOSU'].str.contains('TABLET|CAPSULE|PILL|TABS',na=False),'STD_UNIT']='TABLET'
cm_std_drugs.loc[cm_std_drugs['CMDOSU'].str.contains('SPRAY|PUFF',na=False),'STD_UNIT']='PUFF'
cm_std_drugs.loc[cm_std_drugs['CMDOSU'].str.contains('DROP|gtt',na=False),'STD_UNIT']='DROP'
cm_std_drugs.loc[cm_std_drugs['CMDOSU'].str.contains(r'\bIU\b',na=False),'STD_UNIT']='U'

# mU is million units for benzathyl penicillin
cm_std_drugs.loc[cm_std_drugs['CMDOSU'].str.contains(r"\bMU\b|10\^6 U",na=False),'STD_UNIT']='mU'

# These datapoints are not relevant, as they are either oxygen/anesthetic gas or saline solution
cm_std_drugs.loc[cm_std_drugs['CMDOSU'].str.contains(r'\bL\b|L/min|spores|mg/mL|VIAL|\%|Tbsp',na=False),'STD_UNIT']=np.nan

cm_std_drugs=cm_std_drugs[~cm_std_drugs['STD_UNIT'].isna()]

###########
### STANDARDISE ROUTE OF ADMINISTRATION
cm_std_drugs.loc[cm_std_drugs['CMROUTE'].str.contains(r'INHALATION|RESPIRATORY|INHALALATION',na=False),'STD_ROUTE']='INHALATION'

## One datapoint where emtricitabine was administered (only oral administration)
cm_std_drugs.loc[cm_std_drugs['CMROUTE'].str.contains(r'UNKNOWN',na=False),'STD_ROUTE']='ORAL'

## Check datapoints. where original data is missing the route of administration
#### ALWAYS CHECK THIS DATAFRAME IF ALL DRUGS ARE ORALLY TAKEN!
route_nan_df=cm_std_drugs[cm_std_drugs['STD_ROUTE'].isna()]
## If yes, then set route to oral
cm_std_drugs.loc[cm_std_drugs['STD_ROUTE'].isna(),'STD_ROUTE']='ORAL'

###########
### STANDARDISE DOSE FREQUENCY
## Only standardise frequencies, which don't contain PRN (taken as needed), as in that case the real number of doses is unknown
terms=r'QD|ONCE|Q24H|QN|STAT|OTHER|OVER 6 HOURS|QAM|QM|OVER 4 HOURS|OVER 8HRS, EVERY 16HRS ALTERNATE RINGER|OVER 24 HOURS|GIVEN BETWEEN AND AFTER RECEIVED PACKED RED BLOOD|\+DS|QH|NOCTE'
cm_std_drugs.loc[cm_std_drugs['CMDOSFRQ'].str.contains(terms,na=False)&\
                ~cm_std_drugs['CMDOSFRQ'].str.contains('PRN',na=False),'STD_FREQ']='1 TIME PER DAY'

cm_std_drugs.loc[cm_std_drugs['CMDOSFRQ'].str.contains('BID|Q12H|TWICE STAT',na=False)&\
                ~cm_std_drugs['CMDOSFRQ'].str.contains('PRN',na=False),'STD_FREQ']='2 TIMES PER DAY'

cm_std_drugs.loc[cm_std_drugs['CMDOSFRQ'].str.contains('TID|Q8H|TDS|ONCE, THEN 12 HOURLY, THEN 8 HOURLY',na=False)&\
                ~cm_std_drugs['CMDOSFRQ'].str.contains('PRN',na=False),'STD_FREQ']='3 TIMES PER DAY'                

cm_std_drugs.loc[cm_std_drugs['CMDOSFRQ'].str.contains('QID|Q6H|QDS|6 HOURLY',na=False)&\
                ~cm_std_drugs['CMDOSFRQ'].str.contains('PRN',na=False),'STD_FREQ']='4 TIMES PER DAY'

cm_std_drugs.loc[cm_std_drugs['CMDOSFRQ'].str.contains('Q4H',na=False)&\
                ~cm_std_drugs['CMDOSFRQ'].str.contains('PRN',na=False),'STD_FREQ']='6 TIMES PER DAY'         

cm_std_drugs.loc[cm_std_drugs['CMDOSFRQ'].str.contains('Q2H|Q1H-Q2H',na=False)&\
                ~cm_std_drugs['CMDOSFRQ'].str.contains('PRN',na=False),'STD_FREQ']='12 TIMES PER DAY' 

cm_std_drugs.loc[cm_std_drugs['CMDOSFRQ'].str.contains('EVERY WEEK',na=False)&\
                ~cm_std_drugs['CMDOSFRQ'].str.contains('PRN',na=False),'STD_FREQ']='1 TIME PER WEEK'                  

cm_std_drugs.loc[cm_std_drugs['CMDOSFRQ'].str.contains('3 TIMES PER WEEK ON MONDAY, WEDNESDAY, FRIDAY',na=False)&\
                ~cm_std_drugs['CMDOSFRQ'].str.contains('PRN',na=False),'STD_FREQ']='3 TIMES PER WEEK' 

cm_std_drugs.loc[cm_std_drugs['CMDOSFRQ'].str.contains(r'Q3M|12-WEEKLY|3/12',na=False)&\
                ~cm_std_drugs['CMDOSFRQ'].str.contains('PRN',na=False),'STD_FREQ']='EVERY 12 WEEKS'          

cm_std_drugs.loc[cm_std_drugs['CMDOSFRQ'].str.contains(r'6 DAYS PER WEEK|6TIMES PER WEEK|6 DAYS WEEKLY|6 TIMES WEEKLY|6 DAYS/ WEEK|6 DAYS / WEEK|6 DAYS/WEEK',na=False)&\
                ~cm_std_drugs['CMDOSFRQ'].str.contains('PRN',na=False),'STD_FREQ']='6 TIMES PER WEEK' 


cm_std_drugs.loc[cm_std_drugs['CMDOSFRQ'].str.contains(r'THREE TIMES A DAY /AS NEEDED|3 TIMES DAILY/PRN',na=False)&\
                ~cm_std_drugs['CMDOSFRQ'].str.contains('PRN',na=False),'STD_FREQ']='TID PRN'    

cm_std_drugs.loc[cm_std_drugs['CMDOSFRQ'].str.contains(r'WHEN NECCESSARY',na=False)&\
                ~cm_std_drugs['CMDOSFRQ'].str.contains('PRN',na=False),'STD_FREQ']='PRN' 
            

## '3 TIMES PER MONTH' is probably a typo, as this frequency was taken down for depo-medroxyprogersterone IM injections,
#   which are to inject every 3 months
cm_std_drugs.loc[cm_std_drugs['CMDOSFRQ'].str.contains(r'Q3M|12-WEEKLY|3/12|3 TIMES PER MONTH',na=False)&\
                ~cm_std_drugs['CMDOSFRQ'].str.contains('PRN',na=False),'STD_FREQ']='EVERY 12 WEEKS'    

cm_std_drugs.loc[cm_std_drugs['CMDOSFRQ'].str.contains(r'Q3M|12-WEEKLY|3/12',na=False)&\
                ~cm_std_drugs['CMDOSFRQ'].str.contains('PRN',na=False),'STD_FREQ']='EVERY 12 WEEKS'                

cm_std_drugs.loc[cm_std_drugs['CMDOSFRQ'].str.contains('Q2M',na=False)&\
                ~cm_std_drugs['CMDOSFRQ'].str.contains('PRN',na=False),'STD_FREQ']='EVERY 8 WEEKS' 

cm_std_drugs.loc[cm_std_drugs['CMDOSFRQ'].str.contains('QOD',na=False)&\
                ~cm_std_drugs['CMDOSFRQ'].str.contains('PRN',na=False),'STD_FREQ']='EVERY OTHER DAY' 

## Check the frequencies and 
c=cm_std_drugs['CMDOSFRQ'].value_counts(dropna=False)
d=cm_std_drugs['STD_FREQ'].value_counts(dropna=False)
d.sort_index()

1 TIME PER DAY          2969
1 TIME PER WEEK            3
12 TIMES PER DAY           2
2 TIMES PER DAY         1160
3 TIMES PER DAY         1491
3 TIMES PER WEEK           2
4 TIMES PER DAY          148
4 TIMES PER WEEK           1
4 TIMES PER WEEK PRN       1
5 TIMES PER DAY            5
6 TIMES PER DAY            3
BID PRN                    6
EVERY 10 WEEKS             2
EVERY 12 WEEKS            17
EVERY 6 WEEKS             40
EVERY 8 WEEKS            144
EVERY OTHER DAY            4
PRN                       57
Q12H PRN                   4
Q6H PRN                    1
Q8H PRN                   11
QD PRN                    19
QID PRN                   12
THRICE                     3
TID PRN                  325
TWICE                      3
Name: STD_FREQ, dtype: int64

### Extract the applied concentrations where one cell contains data for multiple applied drugs with their respective concentration <br>
- ### i.e. (Amoxicillin 250 mg Clavulanic acid 125 mg -> Extract drog doses)

In [20]:
## Drop temporarily "CLAVULANIC-ACID" from drugs dictionary, in order to standardise to "CLAVULANIC ACID"
drugs.drugdict.pop("CLAVULANIC-ACID", None)

## Subset dataframe to rows where the cells containing the applied drugs also contain dose information (MG (mg), MCG (ug))
b=cm_std_drugs[cm_std_drugs['STD_DRUGS'].str.contains(r'MCG|MG|\bmcg\b|\bmg\b',na=False)]

## Create column to collect doses and measurement unit of multiple drugs that are contained in one row
b['STD_DOSE_MULTIPLE_DRUGS']=''
b['STD_UNIT_MULTIPLE_DRUGS']=np.nan

## Loop over rows that contain multiple drugs in one cell (== STD_DRUGS_REPLACED columns contains a ',' as there are multiple standardised 
#  drugs)
for row in b.loc[b['STD_DRUGS_REPLACED'].str.contains(',',na=False),'STD_DRUGS'].index:
    
    ## Split the text containing the applied drugs on miligram, ug (MG,MCG) & other conjunctions (and,:), to get substring 
    #  containing drug names and their concentration 
    # (i.e 'AMOXICILLIN 250 MG CLAVULANIC ACID 150 MG -> ['AMOXICILLIN 250','CLAVULANIC ACID 150'])
    drugs_with_cc=re.split(r'AND|\:|MG|MCG',b.loc[row,'STD_DRUGS'])
    drug_ccs=[]

    ## For every row the standardised drug names are already in the STD_DRUGS_REPLACED column. 
    #  For every standardised drug name check if it is contained in the previously splitted subtexts containing drug names & ccs ->
    #  if yes, extract the concentration of the drug
    for std_drug_name in b.loc[row,'STD_DRUGS_REPLACED'].split(','):
        #print(std_drug_name)
        for drug_with_cc in drugs_with_cc:
            for word in drug_with_cc.split(' '):
                #print('standardised word',drugs.standardize([word]),std_drug_name)
                if drugs.standardize([word])==[std_drug_name]:
                    
                    ## If there are no numerical string in the substring,add np.nan
                    if len(re.findall('\d+',drug_with_cc))==0:
                        drug_ccs.append([np.nan])
                    
                    ## If there are numerical strings in the substring, extract & convert them to float       
                    if len(re.findall('\d+',drug_with_cc))>0: 
                        cc=float(re.findall('\d+',drug_with_cc)[0])
                        #print(cc)
                        ## If the drugs were taken as a drug combination in a tablet, multiply the cc with the num of tablets 
                        if b.loc[row,'STD_UNIT']=='TABLET':
                            num_of_tablets=int(b.loc[row,'STD_DOSE'])
                            cc=cc*num_of_tablets
                    
                        drug_ccs.append([cc])
                        #print(cc)
    ## Add calculated drug doses and measurement units to dataframe
    drug_ccs=np.array(drug_ccs).tolist()  
    b.at[row,'STD_DOSE_MULTIPLE_DRUGS']=drug_ccs
    b.at[row,'STD_UNIT_MULTIPLE_DRUGS']='mg'

## Add data to cm_std_drugs dataframe
cm_std_drugs['STD_DOSE_MULTIPLE_DRUGS']=np.nan
cm_std_drugs['STD_UNIT_MULTIPLE_DRUGS']=np.nan

cm_std_drugs.loc[(~(b['STD_DOSE_MULTIPLE_DRUGS']=='')).index,'STD_DOSE_MULTIPLE_DRUGS']=b.loc[~(b['STD_DOSE_MULTIPLE_DRUGS']==''),'STD_DOSE_MULTIPLE_DRUGS']
cm_std_drugs.loc[(~(b['STD_DOSE_MULTIPLE_DRUGS']=='')).index,'STD_UNIT_MULTIPLE_DRUGS']=b.loc[~(b['STD_DOSE_MULTIPLE_DRUGS']==''),'STD_UNIT_MULTIPLE_DRUGS']


########
##  Standardise oral Amoxicillin-clavulanic acid doses. 
# - As clavulanic acid is always 125 mg in oral application, and 200 mg in IV application calculate amox. cc
# - At some datapoints mg is indicated as the unit, but it is a typo probably, as the doses is 1 or 2 -> probably they meant tablet ->
#   for these datapoints don't calculate anything, as the doses are missing here, we only know the number of tablets taken
#   to distinguish these datapoints, set STD_DOSE>10, as if unit is in TABLETs, they number of tablets are probably <10

application_route=['ORAL','INTRAVENOUS']
clav_ccs=[125,200]
for appl_route,clav_cc in zip(application_route,clav_ccs):
    amox_cc=cm_std_drugs.loc[(cm_std_drugs['STD_DRUGS_REPLACED']=='AMOXICILLIN,CLAVULANIC ACID')&\
                    (cm_std_drugs['STD_ROUTE']==appl_route)&(cm_std_drugs['STD_DOSE']>10),'STD_DOSE']-clav_cc

    amoc_clav_ccs=np.array(list(zip(amox_cc,([clav_cc,]*len(amox_cc))))).tolist() 

    for row,cc in zip(amox_cc.index,amoc_clav_ccs):
        cm_std_drugs.at[row,'STD_DOSE_MULTIPLE_DRUGS']=cc
        cm_std_drugs.at[row,'STD_UNIT_MULTIPLE_DRUGS']='mg'


##  Standardise Sulfonyl-urea+ metformin doses. As metformin is always 500 mg in these cases, calculate the dose of the SUR drug from the
#   STD_DOSE columns, as it is given as the sum of the applied SUR + metformin cc
metf_cc=cm_std_drugs[cm_std_drugs['STD_DRUGS_REPLACED'].str.contains(r',METFORMIN',na=False)]

sur_metf_cc=np.array(list(zip(metf_cc['STD_DOSE']-500,([500,]*len(metf_cc))))).tolist() 

for row,cc in zip(metf_cc.index,sur_metf_cc):
        cm_std_drugs.at[row,'STD_DOSE_MULTIPLE_DRUGS']=cc
        cm_std_drugs.at[row,'STD_UNIT_MULTIPLE_DRUGS']='mg'


##  Standardise Ampicillin-cloxacillin/dicloxacillin doses. As Ampicillin is always 250 mg in these cases, calculate the dose of the
#   cloxacillin/dicloxacillin drug from the STD_DOSE columns, as it is given as the sum of the applied Ampicillin-cloxacillin/dicloxacillin cc
amp_cc=cm_std_drugs[cm_std_drugs['STD_DRUGS_REPLACED'].str.contains(r'AMPICILLIN,CLOXACILLIN|AMOXICILLIN,DICLOXACILLIN|CLOXACILLIN,AMPICILLIN',na=False)]

amp_clox_cc=np.array(list(zip(metf_cc['STD_DOSE']-250,([250,]*len(metf_cc))))).tolist() 

for row,cc in zip(amp_cc.index,amp_clox_cc):
        cm_std_drugs.at[row,'STD_DOSE_MULTIPLE_DRUGS']=cc
        cm_std_drugs.at[row,'STD_UNIT_MULTIPLE_DRUGS']='mg'

### Expand the rows containing information about mulitple drugs in one dataframe cell into separate rows

In [22]:
### Expand the rows containing information about mulitple drugs in one dataframe cell into separate rows, 
#   one row for each drug and then drop the rows that contain information about multiple drugs
#   i.e. STD_DRUGS_REPLACED:Amoxicillin, Clavulanic acid, STD_DOSE_MULTIPLE_DRUGS: [200,125] ->
#   make two rows:  row 1: STD_DRUGS_REPLACED Amoxicillin,STD_DOSE:200 ; 
#                   row 2:STD_DRUGS_REPLACED Clavulanic acid,STD_DOSE 125

print(cm_std_drugs.shape)

## Get rows containig information on multiple drugs in one cell
multiple_drugs_per_cell=cm_std_drugs.loc[cm_std_drugs['STD_DRUGS_REPLACED'].str.contains(',',na=False),:]

## Create list to collect the new dataframes into
df_list=[cm_std_drugs]

## Iterate over the rows and create a dataframe with data for one drug per row in it
for row in multiple_drugs_per_cell.index[0:]:
    drugs_list=[np.nan if x=='' else x for x in cm_std_drugs.loc[row,'STD_DRUGS_REPLACED'].split(',')]
    drug_ccs=cm_std_drugs.loc[row,'STD_DOSE_MULTIPLE_DRUGS']

    pat_df=pd.DataFrame(cm_std_drugs.loc[row,:]).T
    pat_df=pd.concat([pat_df]*len(drugs_list),ignore_index=True)  

    pat_df['STD_DRUGS_REPLACED']=drugs_list
    pat_df['STD_DOSE']=drug_ccs
    pat_df['STD_UNIT']='mg'
    pat_df['STD_DOSE_MULTIPLE_DRUGS']=np.nan
    df_list.append(pat_df)

## Concatenate list of ms dataframe and new dataframes created
cm_std_drugs=pd.concat(df_list,ignore_index=True)

## Delete df_list as it is taking lot of memory
del df_list    

## Drop rows that are redundant as their information has been expanded into new rows
cm_std_drugs=cm_std_drugs.loc[~cm_std_drugs['STD_DRUGS_REPLACED'].str.contains(',',na=False),:]

## Set datapoints that contains [nan] as a list to np.nans
cm_std_drugs.loc[cm_std_drugs['STD_DOSE'].apply(lambda x:isinstance(x, list)),'STD_DOSE']=np.nan

print(cm_std_drugs.shape)

(6433, 53)
(6624, 53)


### Save cm dataframe with standardised drug names, dose, dose frequency and application periods

In [23]:
cm_std_drugs.to_csv('../data/out_cm_standardised_with_drugs.csv.gz',compression='gzip')

### Load cm dataframe with standardised drug names, dose, dose frequency and application periods

In [31]:
cm_std_drugs=pd.read_csv('../data/out_cm_standardised_with_drugs.csv.gz',low_memory=False,index_col=0)


### Create dataframe of concomitant drugs __with dosage__
* #### Select only datapoints, where the drugs was taken during study period & dosage frequency is known (frequency is not 'take when necessary'(PRN))


In [25]:

## Multiplication factors to calculate the total daily dose from a single dose and the daily frequency
multiplication_factor={ '1 TIME PER DAY':1,
                        '2 TIMES PER DAY':2,
                        '3 TIMES PER DAY':3,
                        '4 TIMES PER DAY':4,
                        '5 TIMES PER DAY':5,
                        '6 TIMES PER DAY':6,
                        '12 TIMES PER DAY':12} 

## Days of application per week
weekly_freq_dict={  '1 TIME PER WEEK':[0],
                    '2 TIMES PER WEEK':[0,4],
                    '3 TIMES PER WEEK':[0,2,4],
                    '4 TIMES PER WEEK':[0,2,4,6],
                    '5 TIMES PER WEEK':[0,1,2,3,4],
                    '6 TIMES PER WEEK':[0,1,2,3,4,5]}                         

## For drugs taken not every week, calculate the days of application by multiplying the number of weeks with 7
monthly_freq_dict={ 'EVERY 6 WEEKS':6,
                    'EVERY 8 WEEKS':8,
                    'EVERY 10 WEEKS':10,
                    'EVERY 12 WEEKS':12}

## Select data points where only a single drug's information is contained in a cell of 'STD_DRUGS_REPLACED' column.
#  Criteria:
#  - cells that are not empty 
#  - application was during therapy period
#  - dose is known (not PRN== take as necessary)
#  - select onyl rows where the applied drug's concentration is known in mg

d_with_dose=cm_std_drugs[ ~(cm_std_drugs['STD_DRUGS_REPLACED']=='')&\
                (cm_std_drugs['STD_CMENDY']>0)&\
                ~(cm_std_drugs['CMDOSFRQ'].str.contains('PRN',na=False))\
                &(cm_std_drugs['STD_UNIT']=='mg')\
                ]


## Subset dataframe of rows with single drug data to standardised columns with basic descriptive data (studyid, patient id etc)
d_with_dose_std=d_with_dose.loc[:,d_with_dose.columns.str.contains('STD_|USUBJID|STUDYID',na=False)]

## Initialise list to collect results into
concomitant_drugs_with_doses_list=[]

## Loop over all patients and extract the start and stop day,dose, application route for each drug applied over therapy period 
#  into temp_df then concatenate the saved temp_df of each patient to a final dataframe 
## For each drug create a column cm_drugname_application_route (i.e. cm_amoxicillin_oral) 
for pat_id in d_with_dose_std['USUBJID'].unique()[0:]:
    pat_df=d_with_dose_std[d_with_dose_std['USUBJID']==pat_id]
    temp_df=pd.DataFrame(columns=['DAY','STUDYID','USUBJID'],index=np.arange(pat_df['STD_CMSTDY'].min(),pat_df['STD_CMENDY'].max()+1))

    temp_df['USUBJID']=pat_id
    temp_df['STUDYID']=pat_df['STUDYID'].values[0]
    temp_df['DAY']=temp_df.index.tolist()

    ## Loop over each drug patient took
    for drug,drug_df in pat_df.groupby(by='STD_DRUGS_REPLACED'):

        ## If there are mulitple rows in drug_df, that means patient may have changed drug application frequency or 
        # interrupted drug application over therapy period -> loop over rows to catch all this information
        for row in drug_df.index:
            
            ## Extract route and create drug column name cm_drugname_application_route (i.e. cm_amoxicillin_oral) 
            #  plus extract start, stop,dose, freq
            route=drug_df.loc[row,'STD_ROUTE'].lower()
            drug_colname='_'.join(['cmdos',drug.lower(),route])

            ## Create column for drug if it doesn't exist yet
            if drug_colname not in temp_df.columns.tolist():
                temp_df[drug_colname]=np.nan

            freq=drug_df.loc[row,'STD_FREQ']
            dose=drug_df.loc[row,'STD_DOSE']
            start=drug_df.loc[row,'STD_CMSTDY']

            ## Some datapoints go way back before therapy period (i.e. many months-years) -> 
            #  consider relevant period to study 
            if start <-20:
                start=-20
            stop=drug_df.loc[row,'STD_CMENDY']
            
            ## If patient took drug daily and multiple times a day, calculate the daily dose of drug  
            if 'PER DAY' in freq:
                daily_dose=multiplication_factor[freq]*dose
                temp_df.loc[start:stop,drug_colname]=daily_dose
                
            ## If patient didn't take drug daily but every week, calculate the days of application over the appl. period
            if 'PER WEEK' in freq:
                appl_range=stop-start+1
                num_of_weeks=math.ceil(appl_range/7)
                days_of_appl=[]
                for week in range(0,num_of_weeks+1):  
                    days_of_appl.append([day+week*7 for day in weekly_freq_dict[freq]])
                days_of_appl=list(chain(*days_of_appl))         
                days_of_appl=[start+x for x in list(np.array(days_of_appl))]
                days_of_appl=[x for x in days_of_appl if x<=stop] 
                temp_df.loc[days_of_appl,drug_colname]=dose
            
            ## If patient took drug once every x weeks, calculate the days of application
            if 'WEEKS' in freq:
                appl_range=stop-start+1
                num_of_applications=math.ceil((appl_range/7)/monthly_freq_dict[freq])
                days_of_appl=[0+appl*monthly_freq_dict[freq]*7 if num_of_applications>1 else 0 for appl in range(num_of_applications)]
                days_of_appl=[day+start for day in list(np.array(days_of_appl)) if day+start<=stop]
                temp_df.loc[days_of_appl,drug_colname]=dose

            ## If patient took drug every other day, calculate the days if application
            if freq=='EVERY OTHER DAY':
                days_of_appl=np.arange(start,stop+1,2)
                temp_df.loc[days_of_appl,drug_colname]=dose


    ## Drop rows with only NaNs
    nan_rows_mask=~(temp_df.loc[:,temp_df.columns.str.contains('cm',na=False)].isna().all(1))
    temp_df=temp_df[nan_rows_mask]
    
    ## Create cumulative dose column
    for colname in temp_df.columns[temp_df.columns.str.contains('cm',na=False)]:
        temp_df[colname+'_cumul']=temp_df[colname].cumsum()


    ## Drop rows with all NaNs and append temp_df to final list
    temp_df=temp_df.dropna(how='all',axis=1)
    concomitant_drugs_with_doses_list.append(temp_df)

## Concatenate all the dataframes for each patients
cm_temporal_drugs_with_doses=pd.concat(concomitant_drugs_with_doses_list,ignore_index=True)


In [26]:
cm_temporal_drugs_with_doses.shape

(164253, 671)

In [27]:
cm_temporal_drugs_with_doses.to_csv('../data/out_cm_temporal_with_doses.csv.gz',compression='gzip')
del cm_temporal_drugs_with_doses

## Concomitant drugs taken + indications
### 1. Create dataframe (__cm_drugs_with_days_of_appl__) of concomitant drugs __with days of drug application (1 if drug was taken/0 if not)__ <br>
### 2. Create dataframe (__cm_indications_with_days__) of indications for the drugs taken

In [32]:
import warnings
warnings.filterwarnings('ignore')

## Multiplication factors to calculate the total daily dose from a single dose and the daily frequency
multiplication_factor={ '1 TIME PER DAY':1,
                        '2 TIMES PER DAY':2,
                        '3 TIMES PER DAY':3,
                        '4 TIMES PER DAY':4,
                        '5 TIMES PER DAY':5,
                        '6 TIMES PER DAY':6,
                        '12 TIMES PER DAY':12} 

## Days of application per week
weekly_freq_dict={  '1 TIME PER WEEK':[0],
                    '2 TIMES PER WEEK':[0,4],
                    '3 TIMES PER WEEK':[0,2,4],
                    '4 TIMES PER WEEK':[0,2,4,6],
                    '5 TIMES PER WEEK':[0,1,2,3,4],
                    '6 TIMES PER WEEK':[0,1,2,3,4,5]}                         

## For drugs taken not every week, calculate the days of application by multiplying the number of weeks with 7
monthly_freq_dict={ 'EVERY 6 WEEKS':6,
                    'EVERY 8 WEEKS':8,
                    'EVERY 10 WEEKS':10,
                    'EVERY 12 WEEKS':12}

## Select data points where only a single drug's information is contained in a cell of 'STD_DRUGS_REPLACED' column.
#  Criteria:
#  - cells that are not empty 
#  - application was during therapy period
#  - dose is known (not PRN== take as necessary)
#  - select rows where one row contains information about only one drug (STD_UNIT_MULTIPLE_DRUGS is NaN)

d_without_dose=cm_std_drugs[ ~(cm_std_drugs['STD_DRUGS_REPLACED']=='')&\
                (cm_std_drugs['STD_CMENDY']>0)&\
                ~(cm_std_drugs['CMDOSFRQ'].str.contains('PRN',na=False))\
                #&(cm_std_drugs['STD_UNIT']=='mg')\
                ]
       

## Subset dataframe of rows with single drug data to standardised columns with basic descriptive data (studyid, patient id etc)
d_without_dose_std=d_without_dose.loc[:,d_without_dose.columns.str.contains('STD_|USUBJID|STUDYID',na=False)]

## Initialise list to collect results into
concomitant_drugs_with_appl_days_list=[]
concomitant_indications_list=[]


## Loop over all patients and extract the start and stop day,dose, application route for each drug applied over therapy period 
#  into temp_df then concatenate the saved temp_df of each patient to a final dataframe 
## For each drug create a column cm_drugname_application_route (i.e. cm_amoxicillin_oral) 
for pat_id in d_without_dose_std['USUBJID'].unique()[0:]:
    pat_df=d_without_dose_std[d_without_dose_std['USUBJID']==pat_id]
    temp_drug_df=pd.DataFrame(columns=['DAY','STUDYID','USUBJID'],index=np.arange(pat_df['STD_CMSTDY'].min(),pat_df['STD_CMENDY'].max()+1))

    temp_drug_df['USUBJID']=pat_id
    temp_drug_df['STUDYID']=pat_df['STUDYID'].values[0]
    temp_drug_df['DAY']=temp_drug_df.index.tolist()
    temp_ind_df=temp_drug_df.copy()

    ## Loop over each drug patient took
    for drug,drug_df in pat_df.groupby(by='STD_DRUGS_REPLACED'):

        ## If there are mulitple rows in drug_df, that means patient may have changed drug application frequency or 
        # interrupted drug application over therapy period -> loop over rows to catch all this information
        for row in drug_df.index:
            
            ## Extract route and create drug column name cm_drugname_application_route (i.e. cm_amoxicillin_oral) 
            #  plus extract start, stop,dose, freq
            route=drug_df.loc[row,'STD_ROUTE'].lower()
            indication=drug_df.loc[row,'STD_CMINDC']
            drug_colname='_'.join(['cmday',drug.lower(),route])
            ind_colname='_'.join(['cmind',indication])

            ## Create column for drug if it doesn't exist yet
            if drug_colname not in temp_drug_df.columns.tolist():
                temp_drug_df[drug_colname]=np.nan
            
            ## Create column for indication if it doesn't exist yet
            if ind_colname not in temp_ind_df.columns.tolist():
                temp_ind_df[ind_colname]=np.nan

            freq=drug_df.loc[row,'STD_FREQ']
            dose=drug_df.loc[row,'STD_DOSE']
            start=drug_df.loc[row,'STD_CMSTDY']
            ## Some datapoints go way back before therapy period (i.e. many months-years) -> 
            #  consider relevant period to study 
            if start <-20:
                start=-20
            stop=drug_df.loc[row,'STD_CMENDY']
            
            
            ## If patient took drug drug daily
            if 'PER DAY' in freq:
                temp_drug_df.loc[start:stop,drug_colname]=1
                temp_ind_df.loc[start:stop,ind_colname]=1
                
            ## If patient didn't take drug daily but every week, calculate the days of application over the appl. period
            if 'PER WEEK' in freq:
                appl_range=stop-start+1
                num_of_weeks=math.ceil(appl_range/7)
                days_of_appl=[]
                for week in range(0,num_of_weeks+1):  
                    days_of_appl.append([day+week*7 for day in weekly_freq_dict[freq]])
                days_of_appl=list(chain(*days_of_appl))         
                days_of_appl=[start+x for x in list(np.array(days_of_appl))]
                days_of_appl=[x for x in days_of_appl if x<=stop] 
                temp_drug_df.loc[days_of_appl,drug_colname]=1
                temp_ind_df.loc[days_of_appl,ind_colname]=1
            
            ## If patient took drug once every x weeks, calculate the days of application
            if 'WEEKS' in freq:
                appl_range=stop-start+1
                num_of_applications=math.ceil((appl_range/7)/monthly_freq_dict[freq])
                days_of_appl=[0+appl*monthly_freq_dict[freq]*7 if num_of_applications>1 else 0 for appl in range(num_of_applications)]
                days_of_appl=[day+start for day in list(np.array(days_of_appl)) if day+start<=stop]
                temp_drug_df.loc[days_of_appl,drug_colname]=1
                temp_ind_df.loc[days_of_appl,ind_colname]=1

            ## If patient took drug every other day, calculate the days if application
            if freq=='EVERY OTHER DAY':
                days_of_appl=np.arange(start,stop+1,2)
                temp_drug_df.loc[days_of_appl,drug_colname]=1
                temp_ind_df.loc[days_of_appl,ind_colname]=1


    ## Drop rows with only NaNs
    nan_rows_mask=~(temp_drug_df.loc[:,temp_drug_df.columns.str.contains('cm',na=False)].isna().all(1))
    temp_drug_df=temp_drug_df[nan_rows_mask]

    nan_rows_mask=~(temp_ind_df.loc[:,temp_ind_df.columns.str.contains('cmind',na=False)].isna().all(1))
    temp_ind_df=temp_ind_df[nan_rows_mask]

    ## Create cumulative dose column
    for colname in temp_drug_df.columns[temp_drug_df.columns.str.contains('cm',na=False)]:
        temp_drug_df[colname+'_cumul']=temp_drug_df[colname].cumsum()

    ## Drop rows with all NaNs and append temp_df to final list
    temp_drug_df=temp_drug_df.dropna(how='all',axis=0)
    temp_ind_df=temp_ind_df.dropna(how='all',axis=0)
    concomitant_drugs_with_appl_days_list.append(temp_drug_df)
    concomitant_indications_list.append(temp_ind_df)


## Concatenate all the dataframes for each patients
#cm_temporal_drugs_with_days_of_appl=pd.concat(concomitant_drugs_with_appl_days_list,ignore_index=True)
cm_temporal_indications_with_days=pd.concat(concomitant_indications_list,ignore_index=True)




## Save concomitant medication temporal dataframes


In [33]:
#cm_temporal_drugs_with_days_of_appl.to_csv('../data/out_cm_temporal_days_of_application.csv.gz',compression='gzip')
cm_temporal_indications_with_days.to_csv('../data/out_cm_temporal_indications.csv.gz',compression='gzip')
#del cm_temporal_drugs_with_days_of_appl

NameError: name 'cm_temporal_drugs_with_days_of_appl' is not defined